In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, classification_report, hamming_loss
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score
import numpy as np
from sklearn.metrics import multilabel_confusion_matrix
import os

In [2]:
f = files.upload()

Saving all_data.csv to all_data (1).csv


In [3]:
# data read and extract feature matrix and target variable
data = pd.read_csv("all_data.csv")

x = data["sequence"]
y = data["goa"].apply(
    lambda k: [item for item in k.strip("[]").replace("'", "").split(', ')
               if item != '']).values
seq_list = x.to_list()

# Encoding sequences
def compute_kmer_frequencies(sequence, k) -> dict:
    kmer_counts = {}
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        kmer_counts[kmer] = kmer_counts.get(kmer, 0) + 1

    total_kmers = sum(kmer_counts.values())
    kmer_frequencies = {kmer: count / total_kmers for kmer, count in
                        kmer_counts.items()}
    return kmer_frequencies

kmer_freq = [compute_kmer_frequencies(seq, 1) for seq in seq_list]

# Transforming the k-mer Frequencies into a Feature Matrix:
vector = DictVectorizer(sparse=False)
X_vectorized = vector.fit_transform(kmer_freq)


#---------------#
# label info
label_list = []  # all labels
label_count = 0  # number of labels
label_dict = {}  # no of instances per label
for lst in y:
    for label in lst:
        if label not in label_list:
            label_list.append(label)
            label_dict[label] = 0
            label_count += 1
for lst in y:
    for label in lst:
        label_dict[label] += 1

sorted_labels_by_instance_numbers = sorted(label_dict.items(), key=lambda k: k[1])
# print(sorted_labels_by_instance_numbers)

label_list_with_less_than_3_instances =[]
for key in label_dict:
    if label_dict[key] < 3:
        label_list_with_less_than_3_instances.append(key)

print(len(label_list_with_less_than_3_instances)) # 7777 - sp.db

734


In [4]:
# Splitting the Dataset
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y,
                                                    test_size=0.2,
                                                    random_state=1)


# Transforming target variable to a multilabel format
mlb = MultiLabelBinarizer(classes=label_list)
y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

# Standardization of feature values
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# labels not present in the training data set
y_train_labels = mlb.inverse_transform(y_train)
y_test_labels = mlb.inverse_transform(y_test)

# Convert to sets for easier comparison
train_label_set = {label for labels in y_train_labels for label in labels}
test_label_set = {label for labels in y_test_labels for label in labels}

# Find labels present in test set but not in training set
missing_labels = test_label_set - train_label_set

print(len(missing_labels))
print("Labels present in test but missing in train:", missing_labels)

11
Labels present in test but missing in train: {'GO:0099040', 'GO:0140115', 'GO:0090554', 'GO:0099038', 'GO:0047484', 'GO:0046865', 'GO:1905039', 'GO:2001225', 'GO:0098908', 'GO:0046943', 'GO:0140328'}


In [ ]:
# Creating an SVM classifier
svm_classifier = OneVsRestClassifier(SVC(kernel='rbf', gamma=0.2))
# gamma = 0.1 -> 0.60
# gamma = 0.2 -> 0.82
# gamma = 0.5 -> 0.91

# Training the SVM model on the training data
svm_classifier.fit(X_train, y_train)

# Making predictions on the test data
y_pred = svm_classifier.predict(X_test)

In [ ]:
svm_classifier1 = OneVsRestClassifier(SVC(kernel='poly', degree=3, coef0=1))
svm_classifier1.fit(X_train, y_train)
y_pred_1 = svm_classifier1.predict(X_test)

In [22]:
# Calculating perfomance evaluation metrics Accuracy:
accuracy_rbf = accuracy_score(y_test, y_pred)
accuracy_poly = accuracy_score(y_test, y_pred_1)
loss_rbf = hamming_loss(y_test, y_pred)
loss_poly = hamming_loss(y_test, y_pred_1)

macro_precision = precision_score(y_test, y_pred, average='macro',zero_division=0)
macro_recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
micro_precision = precision_score(y_test, y_pred, average='micro',zero_division=0)
micro_recall = recall_score(y_test, y_pred, average='micro',zero_division=0)

print(f'Hamming Loss for RBF kernel: {loss_rbf}')  # Hamming Loss: 9.219777437219492e-05
print(f'Accuracy for RBF kernel: {accuracy_rbf}')

print(f'Hamming Loss for Polynomial kernel: {loss_poly}')  # Hamming Loss: 9.219777437219492e-05
print(f'Accuracy for Polynomial kernel: {accuracy_poly}')

print("Macro-averaged Precision for RBF:", macro_precision)
print("Macro-averaged Recall-RBF:", macro_recall)
print("Micro-averaged Precision-RBF:", micro_precision)
print("Micro-averaged Recall-RBF:", micro_recall)

Hamming Loss for RBF kernel: 0.0003237858472455233
Accuracy for RBF kernel: 0.8293650793650794
Hamming Loss for Polynomial kernel: 0.0010336410403199517
Accuracy for Polynomial kernel: 0.49867724867724866
Macro-averaged Precision for RBF: 0.6252956934260124
Macro-averaged Recall-RBF: 0.6010787801205426
Micro-averaged Precision-RBF: 0.9994182664339732
Micro-averaged Recall-RBF: 0.8918057100482017


In [ ]:
# Accuracy Score for each of the labels

def calculate_label_accuracy(y_true, y_pred ):
    TP = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    TN = np.sum((y_true == 0) & (y_pred == 0), axis=0)
    FP = np.sum((y_true == 0) & (y_pred == 1), axis=0)
    FN = np.sum((y_true == 1) & (y_pred == 0), axis=0)


    # Calculate accuracy for each label
    accuracy = (TP + TN) / (TP + FP + FN + TN)
    return accuracy

label_accuracies = calculate_label_accuracy(y_test, y_pred)

for label, accuracy in zip(mlb.classes_, label_accuracies):
    print(f"Accuracy for label {label}: {accuracy}\n")


In [ ]:
# Precision and recall for each of the labels
precisions = precision_score(y_test, y_pred, average=None, zero_division= 0)
recalls = recall_score(y_test, y_pred, average=None, zero_division= 0)

label_precisions = {label: precision for label, precision in zip(mlb.classes_, precisions)}
label_recalls = {label: recall for label, recall in zip(mlb.classes_, recalls)}

for label in mlb.classes_:
    print(f"Label: {label}")
    print(f" Precision: {label_precisions[label]}")
    print(f" Recall: {label_recalls[label]}\n")

In [13]:
# Plot Confusion Matrix for each label
confusion_matrices = multilabel_confusion_matrix(y_test, y_pred)
for i, cm in enumerate(confusion_matrices):
    label_name = mlb.classes_[i]
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix for {label}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.xticks(ticks=[0.5, 1.5], labels=['Negative', 'Positive'])
    plt.yticks(ticks=[0.5, 1.5], labels=['Negative', 'Positive'])
    plt.show()

In [ ]:
# Report for RBF kernel
r_rbf = classification_report(y_test, y_pred, zero_division=0, target_names= label_list)
print(r_rbf)

In [ ]:
# Report for Polynomial kernel
r_ply = classification_report(y_test, y_pred_1, zero_division=0, target_names= label_list)
print(r_ply)

In [ ]:
for idx in range(len(y_pred)):
    actual_labels = [mlb.classes_[i] for i, val in enumerate(y_test[idx]) if val == 1]
    predicted_labels = [mlb.classes_[i] for i, val in enumerate(y_pred[idx]) if val == 1]

    print(f"Instance {idx+1}:")
    print(f" Actual Labels: {actual_labels}")
    print(f" Predicted Labels: {predicted_labels}\n")